# Deploy Order Agent to Amazon Bedrock AgentCore Runtime

This tutorial series builds a **3-agent e-commerce shopping assistant**. In this notebook, you deploy the Order Agent—an A2A server that queries customer orders from Amazon DynamoDB and is invoked by a Orchestrator agent.

**Notebook 2 of 3** in the multi-agent orchestration series.

### What You'll Build

An **Order Agent** built with Strands Agents and deployed to Amazon Bedrock AgentCore that:
- Queries customer orders from Amazon DynamoDB via Model Context Protocol (MCP)
- Responds to A2A protocol requests from other agents on port 9000
- Uses async lifespan pattern for MCP subprocess initialization
- Advertises its capabilities via an agent card for runtime discovery

### How You'll Build It

Using Strands Agents SDK with MCP client and Amazon Bedrock AgentCore:
- Create DynamoDB table and seed with sample order data
- Use Strands Agents `MCPClient` with stdio transport to run `awslabs-dynamodb-mcp-server` as a subprocess
- Use async lifespan pattern to handle MCP startup delays without failing health checks
- Deploy to AgentCore runtime with IAM execution role including DynamoDB permissions
- Store the runtime URL in SSM Parameter Store for multi-agent discovery

## Architecture Overview

```
User Request
     │
     ▼
┌──────────────────┐
│   Orchestrator   │
│ (HTTP protocol)  │
└────────┬─────────┘
      │ A2A Protocol (SigV4 Auth)
      ├───────────────────────────────┬
      ▼                               ▼
┌─────────────┐                 ┌─────────────┐
│ Order Agent │  ◄── You are   │Product Agent│
│ (A2A Server)│      here      │(A2A Server) │
└─────┬───────┘   (Notebook 2) └─────┬───────┘
      │                               │
      ▼                               ▼
┌─────────────┐                 ┌─────────────┐
│  DynamoDB   │                 │Product      │
│  (Orders)   │                 │Catalog API  │
└─────────────┘                 └─────────────┘
```

## Prerequisites

- [AWS CLI](https://aws.amazon.com/cli/) installed and configured
- Python 3.10 or higher
- Docker or Podman installed (only for local container builds; not required if using CodeBuild)
- Claude Sonnet 4 model access in Amazon Bedrock
- **Notebook 01 completed**: Product Agent deployed to Amazon Bedrock AgentCore
- IAM permissions to:
  - Create IAM roles and policies
  - Create Amazon ECR repositories
  - Deploy Amazon Bedrock AgentCore runtimes
  - Create and write to Amazon DynamoDB tables
  - Write to AWS Systems Manager Parameter Store
- Local notebook dependencies: `pip install -r requirements.txt`

In [ ]:
# Install notebook dependencies
!pip install -r requirements.txt

In [ ]:
import json
import os
import sys
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

import boto3

# Store the notebook's directory - must be defined before any os.chdir() calls
NOTEBOOK_DIR = Path.cwd()

# Import utility modules
from utils.config import (
    SSM_ORDER_AGENT_URL,
    SSM_ORDERS_TABLE,
    ORDER_AGENT_NAME,
    ORDER_ROLE_NAME,
    DYNAMODB_TABLE_NAME,
)
from utils.iam import create_agentcore_role
from utils.dynamodb_setup import create_orders_table, seed_orders, delete_orders_table
from utils.ssm_helpers import store_ssm_parameter, delete_parameters

# Get AWS account information
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
region = "us-west-2"

print(f"AWS Account: {account_id}")
print(f"Region: {region}")

## Step 1: Create Order Agent with MCP Integration

The Order Agent is a **specialist agent** that other agents call to query customer order data. It uses the Strands Agents `MCPClient` with stdio transport to run `awslabs-dynamodb-mcp-server` as a subprocess.

**MCP Integration Architecture**

```
┌─────────────────────────────────────────────────────────────┐
│ AgentCore Container                                         │
│  ┌──────────────────────────────────────────────────────┐  │
│  │ A2A Server (FastAPI on port 9000)                    │  │
│  │  ┌────────────────────────────────────────────────┐  │  │
│  │  │ Strands Agent                                  │  │  │
│  │  │  ├─ MCPClient (stdio transport)                │  │  │
│  │  │  │   └─ awslabs-dynamodb-mcp-server subprocess │  │  │
│  │  │  │        └─ DynamoDB API calls                │  │  │
│  │  │  └─ BedrockModel (Claude Sonnet)               │  │  │
│  │  └────────────────────────────────────────────────┘  │  │
│  └──────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘
```

| Component | Role |
|-----------|------|
| `MCPClient` | Strands Agents SDK class that manages MCP server connections |
| stdio transport | Runs MCP server as subprocess, communicates via stdin/stdout |
| `awslabs-dynamodb-mcp-server` | Pre-built MCP server providing DynamoDB query/scan tools |
| Async lifespan | Delays MCP subprocess startup until after server binds to port 9000 |

**Key pattern:** The async lifespan handler launches the MCP subprocess *after* the A2A server binds to port 9000, ensuring health checks pass before slow subprocess initialization begins.

### Set Up DynamoDB Orders Table

Create the DynamoDB table and seed it with sample order data before deploying the agent.

In [ ]:
# Create DynamoDB table
print("Creating DynamoDB orders table...")
result = create_orders_table(table_name=DYNAMODB_TABLE_NAME, region=region)
table_name = DYNAMODB_TABLE_NAME
print(f"Table: {table_name} ({result['status']})")

In [ ]:
# Seed sample orders
orders_file = NOTEBOOK_DIR / "sample_data" / "orders.json"
print(f"Loading sample orders from: {orders_file}")
result = seed_orders(table_name=table_name, json_file=orders_file, region=region)
print(f"Seeded {result['count']} orders")

In [ ]:
# Store table name in SSM
result = store_ssm_parameter(
    param_name=SSM_ORDERS_TABLE,
    value=table_name,
    region=region,
)
print(result["message"])
print(f"Parameter version: {result['version']}")

### Code Structure

The following cell creates `order_agent/a2a_server.py`:

| Section | What It Does |
|---------|--------------|
| Configuration | Constants for port (9000), SSM parameter paths, MCP timeouts, and system prompt |
| `ToolLoggingHandler` | Callback that logs tool invocations to CloudWatch for debugging |
| `get_table_name()` | Retrieves DynamoDB table name from environment or SSM Parameter Store |
| `get_runtime_url()` | Retrieves AgentCore runtime URL for agent card generation |
| Agent creation | Creates `order_agent` with empty `tools=[]` for fast module import |
| `server_lifespan()` | Async context manager that launches MCP subprocess after server binds to port |
| A2A Server | Wraps agent with `A2AServer`, mounts on FastAPI app with `/ping` health check |

In [ ]:
%%writefile order_agent/a2a_server.py
"""Order Agent deployed to Amazon Bedrock AgentCore with A2A protocol support.

Queries customer orders from Amazon DynamoDB using the awslabs.dynamodb-mcp-server
MCP tool. Uses async lifespan pattern for MCP subprocess initialization to ensure
health checks pass before tool loading begins.

Key concepts demonstrated:
- A2A protocol server for multi-agent orchestration (port 9000)
- MCP client with stdio transport for DynamoDB access
- Two-phase initialization: fast startup, then async tool loading
- Graceful degradation if MCP subprocess fails
"""

import asyncio
import json
import logging
import os
from contextlib import asynccontextmanager
from typing import Any

import boto3
import uvicorn
from fastapi import FastAPI
from mcp import StdioServerParameters, stdio_client
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer
from strands.telemetry import StrandsTelemetry
from strands.tools.mcp import MCPClient

# --- Logging ---

logging.basicConfig(level=logging.INFO)
logging.getLogger("strands").setLevel(logging.INFO)
logging.getLogger("a2a").setLevel(logging.DEBUG)
logging.getLogger("strands.multiagent.a2a").setLevel(logging.DEBUG)
logger = logging.getLogger(__name__)

# Enable distributed tracing with AWS X-Ray
StrandsTelemetry().setup_otlp_exporter()

# --- Configuration ---

PORT = 9000  # A2A protocol uses port 9000 (HTTP protocol uses 8080)
SSM_ORDER_AGENT_URL = "/ecommerce-assistant/order-agent-url"
SSM_ORDERS_TABLE = "/ecommerce-assistant/orders-table"

# Timeouts prevent container hangs if MCP server fails to start
MCP_STARTUP_TIMEOUT = 10.0  # Max seconds to wait for subprocess
MCP_TOOLS_TIMEOUT = 5.0     # Max seconds to fetch tool definitions

SYSTEM_PROMPT_TEMPLATE = """You are an Order Agent for an e-commerce shopping assistant.

Query the "{table_name}" DynamoDB table for order information.
Return order details including product_ids so the Orchestrator can fetch product details."""


# --- Callback Handler ---
class ToolLoggingHandler:
    """Log tool invocations to CloudWatch for debugging multi-agent workflows.
    
    Tracks which tools the agent calls and with what inputs. Useful for:
    - Debugging why an agent gave a certain response
    - Monitoring A2A calls in distributed systems
    - Building observability dashboards
    """

    def __init__(self) -> None:
        self.logged_tool_ids: set[str] = set()  # Prevent duplicate logs
        self.tool_count = 0

    def __call__(self, **kwargs: Any) -> None:
        message = kwargs.get("message", {})
        if isinstance(message, dict) and message.get("role") == "assistant":
            for content in message.get("content", []):
                if isinstance(content, dict):
                    tool_use = content.get("toolUse")
                    if tool_use:
                        tool_id = tool_use.get("toolUseId")
                        if tool_id and tool_id not in self.logged_tool_ids:  # Prevent duplicate logs
                            self.logged_tool_ids.add(tool_id)
                            self.tool_count += 1
                            tool_name = tool_use.get("name", "Unknown")
                            tool_input = tool_use.get("input", {})
                            logger.info(f"=== TOOL #{self.tool_count}: {tool_name} ===")
                            logger.info(f"TOOL INPUT: {json.dumps(tool_input)}")

        if kwargs.get("complete") and kwargs.get("data"):
            logger.info(f"=== COMPLETE: {len(kwargs.get('data', ''))} chars ===")


# --- Helper Functions ---
def get_table_name() -> str:
    """Get DynamoDB table name from environment variable or SSM Parameter Store.
    
    Checks ORDERS_TABLE env var first (useful for local dev), then falls back
    to SSM (standard for AgentCore deployments where secrets are stored in SSM).
    """
    if table_name := os.environ.get("ORDERS_TABLE"):
        logger.info(f"Using table from env: {table_name}")
        return table_name

    try:
        ssm = boto3.client("ssm")
        response = ssm.get_parameter(Name=SSM_ORDERS_TABLE, WithDecryption=True)
        table_name = response["Parameter"]["Value"]
        logger.info(f"Using table from SSM: {table_name}")
        return table_name
    except Exception as e:
        logger.error(f"Could not get table name: {e}")
        raise


def get_runtime_url() -> str | None:
    """Get AgentCore runtime URL for agent card generation.
    
    The runtime URL is needed for the A2A agent card (/.well-known/agent-card.json)
    so other agents can discover and call this agent. Without it, the agent card
    will have an internal URL that won't work for external A2A calls.
    """
    if url := os.environ.get("ORDER_AGENT_URL"):
        logger.info(f"Using runtime URL from env: {url}")
        return url

    try:
        ssm = boto3.client("ssm")
        response = ssm.get_parameter(Name=SSM_ORDER_AGENT_URL, WithDecryption=True)
        url = response["Parameter"]["Value"]
        logger.info(f"Using runtime URL from SSM: {url}")
        return url
    except Exception as e:
        logger.warning(f"Could not get runtime URL: {e}")
        return None


# --- Agent Creation ---
# Create agent with tools=[] for fast module import. 
# MCP tools are added later in server_lifespan() after the server binds to port 9000.
# MCP subprocess startup can take several seconds, so we defer it to lifespan.

region = boto3.session.Session().region_name or "us-west-2"

order_agent = Agent(
    name="Ecommerce_Order_Agent",
    description="Looks up order information from DynamoDB for customers",
    system_prompt="Order agent initializing...",  # Updated in server_lifespan
    model=BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
        region_name=region,
    ),
    tools=[],  # Empty - MCP tools added in server_lifespan after startup
    callback_handler=ToolLoggingHandler(),
)
logger.info(f"Agent created: {order_agent.name}")

# MCP connection handle - initialized in server_lifespan, cleaned up on shutdown
mcp_tool_connection: MCPClient | None = None


# --- Server Lifespan (startup and shutdown) ---
@asynccontextmanager
async def server_lifespan(app: FastAPI):
    """Initialize MCP tools after server starts, clean up on shutdown.
    
    This runs AFTER uvicorn binds to port 9000, so AgentCore health checks pass immediately.
    The MCP subprocess starts here, not during module import.
    
    Sequence:
    1. Module imports → agent created with tools=[]
    2. uvicorn binds to port 9000 → /ping endpoint ready
    3. AgentCore health checks pass → container marked healthy
    4. THIS FUNCTION runs → MCP subprocess starts → tools added to agent
    5. Agent ready to handle A2A requests with DynamoDB access
    """
    global mcp_tool_connection

    logger.info("=== Order Agent Startup ===")

    try:
        table_name = get_table_name()

        # Launch DynamoDB MCP server as subprocess via uvx (auto-installs package)
        # The MCP server provides query/scan tools that the agent uses to read orders
        mcp_tool_connection = MCPClient(
            lambda: stdio_client(
                StdioServerParameters(
                    command="uvx",
                    args=["awslabs.dynamodb-mcp-server@latest"],
                    env={
                        **os.environ,  # Inherit AWS credentials from container's IAM role
                        "AWS_REGION": region,
                        "DDB_TABLE_NAME": table_name,
                    },
                )
            )
        )

        # Start the MCP subprocess with timeout protection
        # run_in_executor allows synchronous MCP server startup to run without blocking the async event loop.
        event_loop = asyncio.get_event_loop()
        start_mcp_subprocess = mcp_tool_connection.__enter__
        await asyncio.wait_for(
            event_loop.run_in_executor(None, start_mcp_subprocess),
            timeout=MCP_STARTUP_TIMEOUT,
        )
        logger.info("MCP tool connection started")

        # Fetch available tools from MCP server 
        mcp_tools = await asyncio.wait_for(
            event_loop.run_in_executor(None, mcp_tool_connection.list_tools_sync),
            timeout=MCP_TOOLS_TIMEOUT,
        )
        logger.info(f"Loaded {len(mcp_tools)} tools from MCP server")

        # Attach tools and update system prompt now that we know the table
        order_agent.tools = mcp_tools
        order_agent.system_prompt = SYSTEM_PROMPT_TEMPLATE.format(table_name=table_name)

        logger.info(f"=== Startup Complete: {order_agent.name} with {len(mcp_tools)} tools ===")

    except asyncio.TimeoutError:
        # Graceful degradation: agent stays healthy but can't query DynamoDB
        logger.error("MCP initialization timed out - agent will operate without tools")
    except Exception as e:
        logger.error(f"Startup error: {e} - agent will operate without tools")

    yield  # Control returns to FastAPI; server is now serving requests

    # Before shutdown, ensure MCP subprocess stops
    logger.info("=== Shutting Down ===")
    if mcp_tool_connection:
        try:
            stop_mcp_subprocess = mcp_tool_connection.__exit__
            stop_mcp_subprocess(None, None, None)
            logger.info("MCP tool connection closed")
        except Exception as e:
            logger.error(f"Error closing MCP: {e}")


# --- FastAPI App and A2A Server ---
runtime_url = get_runtime_url()

a2a_server = A2AServer(
    agent=order_agent,
    host="0.0.0.0",  # Listen on all interfaces 
    port=PORT,
    http_url=runtime_url,  # Used in agent card for A2A discovery
    serve_at_root=True,    # Serve agent card at /.well-known/agent-card.json
)

app = FastAPI(title="Order Agent A2A Server", lifespan=server_lifespan)


@app.get("/ping")
def ping():
    """Health check endpoint for AgentCore container probes."""
    return {"status": "healthy"}


app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=PORT)

In [ ]:
%%writefile order_agent/requirements.txt
strands-agents[a2a,otel]==1.29.0
fastapi==0.115.0
uvicorn==0.34.0
boto3==1.42.60
mcp==1.9.2

---
## Step 2: Deploy to Amazon Bedrock AgentCore

The `bedrock-agentcore-starter-toolkit` handles the deployment pipeline. Behind the scenes, it:

1. **Creates IAM role** — Grants permissions for ECR image pull, Bedrock model invocation, DynamoDB access, and CloudWatch logging
2. **Configures runtime** — Packages agent code with MCP dependencies and sets A2A protocol
3. **Builds container** — Creates Docker image and pushes to Amazon ECR
4. **Launches runtime** — Starts managed container in Amazon Bedrock AgentCore
5. **Stores URL** — Saves runtime endpoint to Parameter Store for discovery

### What Gets Created

| Resource | Name/Path | Purpose |
|----------|-----------|---------|
| DynamoDB Table | `ecommerce-orders` | Stores customer order data with a global secondary index (GSI) on `customer_id` for efficient customer-specific queries |
| IAM Role | `ecommerce-order-agent-role` | Grants the agent runtime permissions to pull container images from Amazon ECR, invoke Amazon Bedrock models, read from DynamoDB, and write logs to Amazon CloudWatch |
| ECR Repository | `ecommerce_order_agent` | Stores the Docker container image that Amazon Bedrock AgentCore pulls to run the agent |
| AgentCore Runtime | `ecommerce_order_agent` | Managed compute environment that runs your containerized A2A server with MCP subprocess|
| SSM Parameters | `/ecommerce-assistant/order-agent-url`, `/ecommerce-assistant/orders-table` | Store the runtime invocation URL and DynamoDB table name so the Orchestrator and agent container can discover this agent's endpoint and configuration |

### Create IAM Role and Configure Runtime

The IAM execution role allows the AgentCore runtime to pull container images, invoke Bedrock models, query DynamoDB, and write logs.

| Parameter | Value | Purpose |
|-----------|-------|---------|  
| `entrypoint` | `a2a_server.py` | Python file that starts the A2A server with async lifespan |
| `protocol` | `A2A` | Enables port 9000 for agent-to-agent messaging |
| `auto_create_ecr` | `True` | Creates ECR repository if it doesn't exist |
| `additional_permissions` | DynamoDB actions | Allows GetItem, Query, Scan on orders table |

In [ ]:
# Create IAM role with DynamoDB permissions
table_arn = f"arn:aws:dynamodb:{region}:{account_id}:table/{table_name}"
index_arn = f"{table_arn}/index/*"

dynamodb_permissions = [
    {
        "Effect": "Allow",
        "Action": [
            "dynamodb:GetItem",
            "dynamodb:Query",
            "dynamodb:Scan",
            "dynamodb:DescribeTable"
        ],
        "Resource": [
            table_arn,
            index_arn
        ]
    }
]

order_role_arn = create_agentcore_role(
    role_name=ORDER_ROLE_NAME,
    account_id=account_id,
    region=region,
    additional_permissions=dynamodb_permissions,
)
print(f"Order Agent Role: {order_role_arn}")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Change to order_agent directory
order_agent_dir = NOTEBOOK_DIR / "order_agent"
os.chdir(order_agent_dir)

# Configure runtime
order_runtime = Runtime()
order_runtime.configure(
    entrypoint="a2a_server.py",
    execution_role=order_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=ORDER_AGENT_NAME,
    protocol="A2A",
)
print("Order Agent runtime configured")

### Launch Agent

`Runtime.launch()` builds the Docker image, pushes it to ECR, and creates the AgentCore runtime. The `auto_update_on_conflict=True` flag updates an existing runtime if one already exists with the same name.

**Note:** First deployment takes 5-10 minutes. Order Agent may take slightly longer than Product Agent due to additional MCP dependencies.

In [ ]:
# Launch Order Agent
print("Launching Order Agent (this may take several minutes)...")
order_launch = order_runtime.launch(auto_update_on_conflict=True)
print(f"Order Agent ARN: {order_launch.agent_arn}")
ORDER_AGENT_ARN = order_launch.agent_arn

### Get Runtime URL

Once the runtime reaches `ACTIVE` or `READY` status, construct the invocation URL from the runtime ARN. This URL is the HTTPS endpoint that the Orchestrator will use to send A2A messages to this agent.

In [ ]:
# Get runtime status and URL
status_response = order_runtime.status()
status = status_response.endpoint.get("status", "")
order_agent_url = None  # Initialize before conditional

print(f"Order Agent Status: {status}")

if status.upper() in ["ACTIVE", "READY"]:
    agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn")
    escaped_arn = quote(agent_runtime_arn, safe="")
    order_agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    print(f"Order Agent URL: {order_agent_url}")
else:
    print(f"WARNING: Order Agent status is {status}")
    print("Expected status: ACTIVE or READY")

### Store Runtime URL in Parameter Store

Store the URL in AWS Systems Manager Parameter Store so other components can discover this agent:
- **Agent container** reads it at startup to populate the agent card's `http_url` field
- **Orchestrator Agent** (Notebook 3) retrieves it to configure `A2AClientToolProvider`

In [ ]:
# Store Order Agent URL in SSM
if status.upper() in ["ACTIVE", "READY"]:
    result = store_ssm_parameter(
        param_name=SSM_ORDER_AGENT_URL,
        value=order_agent_url,
        region=region
    )
    print(result["message"])
    print(f"Parameter version: {result['version']}")
else:
    print("Skipping SSM storage - agent not ready")

# Return to notebook directory
os.chdir(NOTEBOOK_DIR)

In [ ]:
print("=" * 60)
print("Order Agent Deployment Summary")
print("=" * 60)
print(f"Agent Name: {ORDER_AGENT_NAME}")
print(f"Agent ARN: {ORDER_AGENT_ARN}")
print(f"IAM Role: {order_role_arn}")
print(f"Runtime URL: {order_agent_url or 'Not available - agent not ready'}")
print(f"DynamoDB Table: {table_name}")
print(f"SSM Parameters:")
print(f"  - {SSM_ORDER_AGENT_URL}")
print(f"  - {SSM_ORDERS_TABLE}")
print(f"Status: {status}")
print(f"Region: {region}")
print(f"Protocol: A2A (port 9000)")
print(f"MCP Server: awslabs-dynamodb-mcp-server")
print("=" * 60)

---
## Step 3: Test Order Agent

Verify the agent works standalone before integrating with the Orchestrator. The `Runtime.invoke()` method calls the AgentCore runtime directly—useful for testing individual agents without needing another agent to send A2A messages.

**Example queries to try:**
- `"What orders does customer CUST-101 have?"`
- `"Show me details for order ORD-001"`
- `"List all pending orders"`

**Expected response:**
- Agent invokes MCP tools (`query_table` or `scan_table`) from `awslabs-dynamodb-mcp-server`
- Returns JSON with matching orders (order_id, customer_id, product_ids, status, total_amount)
- Response is conversational, summarizing the order results

In [ ]:
# Test Order Agent with A2A protocol payload
test_message = "What orders does customer CUST-101 have?"

# A2A protocol uses JSON-RPC 2.0 format
payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": str(uuid4()),
    "params": {
        "message": {
            "messageId": str(uuid4()),
            "role": "user",
            "parts": [{"kind": "text", "text": test_message}]
        }
    }
}

print("Testing Order Agent...")
print(f"Query: {test_message}")
print("-" * 60)


try:
    response = order_runtime.invoke(payload=payload, session_id=str(uuid4()))
    if "result" in str(response) and "artifacts" in str(response):
        print(response)
    else:
        print("Unexpected response format")
        print(str(response)[:500])
except Exception as e:
    print(f"Error: {e}")

### Verify Agent Card (A2A Discovery)

The `A2AServer` wrapper automatically generates an **A2A-compliant agent card** from your Strands agent. This card is served at `/.well-known/agent-card.json` and enables other agents to discover this agent's capabilities at runtime.

**What A2AServer auto-generates:**
- Agent name and description from `Agent()` constructor
- Skills array from MCP tools loaded during async lifespan startup
- Supported input/output modes and transport protocol
- Runtime URL for A2A messaging

In [ ]:
# Fetch the auto-generated agent card
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests

agent_card_url = f"{order_agent_url}/.well-known/agent-card.json"
credentials = boto3.Session().get_credentials()
request = AWSRequest(method="GET", url=agent_card_url)
SigV4Auth(credentials, "bedrock-agentcore", region).add_auth(request)

response = requests.get(agent_card_url, headers=dict(request.headers))
agent_card = response.json()

print(json.dumps(agent_card, indent=2))

---
## Next Steps

| Notebook | What You'll Build |
|----------|-------------------|
| **3. Deploy Orchestrator** | HTTP-facing agent that routes requests to Product and Order agents via `A2AClientToolProvider` |

---
## Cleanup (Optional)

Run this section to delete all resources created by this notebook.

In [ ]:
# Destroy Order Agent
print("Destroying Order Agent...")
os.chdir(order_agent_dir)
try:
    order_runtime.destroy(delete_ecr_repo=True)
    print("Order Agent destroyed")
except Exception as e:
    print(f"Error: {e}")
os.chdir(NOTEBOOK_DIR)